In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D8 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D8 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D8"
DOCUMENT_NAME = (
    "World Bank — Bhutan - Land Management Project — "
    "Project Information Document (PID), Concept Stage"
)

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_RECORD_COUNT = 49

EXPECTED_CATEGORY_COUNTS = {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5,
}

FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location",
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Category",
    "Topic",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location",
]

DESCRIPTION_DIAGNOSTIC_FIELD = "Description"

MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location",
]

NULLABLE_STRING_FIELDS = [
    "Unit",
    "Qualifier",
    "Reporting Period",
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

BLOCK_FIELDS = [
    "Category",
    "Source Location",
]

MATCHING_WEIGHTS = {
    "topic": 0.65,
    "description": 0.25,
    "reporting_period": 0.10,
}

MATCH_SCORE_THRESHOLD = 0.35

OUTPUT_DIR = Path("outputs_D8_validation_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH, "-", BRANCH_NAME)
print("Expected records:", EXPECTED_RECORD_COUNT)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)


In [ ]:
# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D8_reference_values.csv
#   2) D8_branch_B_parsed_extraction.json
#   3) D8_branch_B_technical_diagnostics.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    name for name in uploaded_files
    if name.lower().endswith(".csv")
]

json_files = [
    name for name in uploaded_files
    if name.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one D8 Stage 1 reference CSV."
    )

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the Branch B parsed "
        "extraction and technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]
PARSED_EXTRACTION_FILE = None
TECHNICAL_DIAGNOSTICS_FILE = None

for file_name in json_files:
    with open(file_name, "r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structurally_evaluable" in obj
        and "record_schema_valid" in obj
        and "valid_json" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify the D8 Branch B parsed extraction."
    )

if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D8 Branch B technical diagnostics."
    )

print("Reference:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Technical diagnostics:", TECHNICAL_DIAGNOSTICS_FILE)


In [ ]:
# ============================================================
# 3. Load inputs and verify identity/provenance
# ============================================================

with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    extraction_content = json.load(f)

with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    technical_diagnostics = json.load(f)

reference_df = pd.read_csv(
    REFERENCE_FILE,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
).replace("", None)

def restore_reference_value(value):
    if value is None:
        return None

    text = str(value).strip()

    if text.casefold() in {"one-quarter", "one-third"}:
        return text

    if re.fullmatch(r"-?\d+(?:\.\d+)?", text):
        try:
            number = float(text)
            return int(number) if number.is_integer() else number
        except ValueError:
            pass

    return value

reference_df["Value"] = reference_df["Value"].map(
    restore_reference_value
)

for artefact_name, artefact in {
    "parsed extraction": extraction_content,
    "technical diagnostics": technical_diagnostics,
}.items():
    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id."
        )
    if artefact.get("branch") != BRANCH:
        raise ValueError(
            f"Unexpected {artefact_name} branch."
        )

extracted_records = extraction_content["records"]
extracted_df = pd.DataFrame(extracted_records)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": PARSED_EXTRACTION_FILE,
    "parsed_extraction_sha256":
        sha256_file(PARSED_EXTRACTION_FILE),
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_FILE,
    "technical_diagnostics_sha256":
        sha256_file(TECHNICAL_DIAGNOSTICS_FILE),
}

print("Reference shape:", reference_df.shape)
print("Extraction shape:", extracted_df.shape)


In [ ]:
# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

schema_validity = bool(structurally_evaluable)

schema_diagnostics = {
    "valid_json": bool(
        technical_diagnostics.get(
            "valid_json",
            False
        )
    ),
    "record_schema_valid": bool(
        technical_diagnostics.get(
            "record_schema_valid",
            False
        )
    ),
    "field_types_valid": bool(
        technical_diagnostics.get(
            "field_types_valid",
            False
        )
    ),
    "structurally_evaluable":
        structurally_evaluable,
    "schema_validity":
        schema_validity,
}

if not structurally_evaluable:
    raise ValueError(
        "D8 Branch B output is not structurally evaluable. "
        "Content-level validation cannot proceed."
    )

print(json.dumps(
    schema_diagnostics,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 5. Verify fixed Stage 1 reference and prepare comparison copy
# ============================================================

reference_schema_exact = (
    reference_df.columns.tolist() == FIELDS
)

reference_record_count_valid = (
    len(reference_df) == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts == EXPECTED_CATEGORY_COUNTS
)

if not reference_schema_exact:
    raise ValueError(
        "D8 Stage 1 reference schema does not match "
        "the fixed field list."
    )

if not reference_record_count_valid:
    raise ValueError(
        f"Expected {EXPECTED_RECORD_COUNT} Stage 1 records; "
        f"found {len(reference_df)}."
    )

if not reference_category_counts_valid:
    raise ValueError(
        "D8 Stage 1 category counts do not match "
        "the fixed design."
    )

missing_extraction_fields = [
    field for field in FIELDS
    if field not in extracted_df.columns
]

extracted_comparison_df = extracted_df.copy(deep=True)

for field in missing_extraction_fields:
    extracted_comparison_df[field] = None

extracted_comparison_df = (
    extracted_comparison_df[FIELDS].copy()
)

reference_comparison_df = (
    reference_df[FIELDS].copy(deep=True)
)

extracted_record_count = len(extracted_comparison_df)

extraction_category_counts = (
    extracted_comparison_df["Category"]
    .value_counts(dropna=False)
    .to_dict()
)

content_diagnostics = {
    "reference_record_count_valid": True,
    "reference_category_counts_valid": True,
    "extraction_record_count_valid":
        extracted_record_count == EXPECTED_RECORD_COUNT,
    "extraction_category_counts_valid":
        extraction_category_counts == EXPECTED_CATEGORY_COUNTS,
    "branch_B_scope_complete":
        technical_diagnostics.get("scope_complete"),
    "branch_B_content_diagnostics":
        technical_diagnostics.get("content_diagnostics"),
}

print("Reference records:", len(reference_comparison_df))
print("Extracted records:", len(extracted_comparison_df))
print("Missing extraction columns:", missing_extraction_fields)


In [ ]:
# ============================================================
# 6. Controlled comparison normalisation
# ============================================================

def is_missing(value):
    if value is None:
        return True

    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def normalise_text(value):
    if is_missing(value):
        return None

    text = str(value)

    text = "".join(
        character
        for character in text
        if unicodedata.category(character) != "Cf"
    )

    text = unicodedata.normalize("NFKC", text)

    text = (
        text.replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
        .replace("–", "-")
        .replace("—", "-")
        .replace("\u00a0", " ")
    )

    text = re.sub(r"\s+", " ", text).strip()
    return text.casefold()


def identity_text(value):
    text = normalise_text(value)

    if text is None:
        return ""

    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def text_similarity(first, second):
    first_text = identity_text(first)
    second_text = identity_text(second)

    if not first_text and not second_text:
        return 1.0

    if not first_text or not second_text:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text,
    ).ratio()


def exact_normalised_text_match(first, second):
    return normalise_text(first) == normalise_text(second)


In [ ]:
# ============================================================
# 7. Controlled D8 equivalence rules
# ============================================================


UNIT_EQUIVALENCE_MAP = {
    "%": "percent",
    "percent": "percent",
    "usd million": "usd million",
    "million usd": "usd million",
    "$ million": "usd million",
    "usd millions": "usd million",
}


def canonical_unit(value):
    text = normalise_text(value)

    if text is None:
        return None

    return UNIT_EQUIVALENCE_MAP.get(text, text)


def canonical_period(value):
    return normalise_text(value)


def canonical_qualifier(value):
    return normalise_text(value)


def canonical_topic(value):
    return normalise_text(value)


def topic_correct(reference_value, extracted_value):
    return (
        canonical_topic(reference_value)
        == canonical_topic(extracted_value)
    )


def unit_correct(reference_value, extracted_value):
    return (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    )


def qualifier_correct(reference_value, extracted_value):
    return (
        canonical_qualifier(reference_value)
        == canonical_qualifier(extracted_value)
    )


def period_correct(reference_value, extracted_value):
    return (
        canonical_period(reference_value)
        == canonical_period(extracted_value)
    )


print(
    "Conservative D8 rules loaded. "
    "No Branch-A-specific Topic aliases are active."
)


In [ ]:
# ============================================================
# 8. Null-safe Value comparison
# ============================================================

def numeric_value(value):
    if is_missing(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    return None


def value_correct(reference_value, extracted_value):

    if is_missing(reference_value) and is_missing(extracted_value):
        return True

    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if reference_number is not None and extracted_number is not None:
        return math.isclose(
            reference_number,
            extracted_number,
            rel_tol=0.0,
            abs_tol=1e-12,
        )


    if isinstance(reference_value, str) and isinstance(extracted_value, str):
        return (
            normalise_text(reference_value)
            == normalise_text(extracted_value)
        )

    return False


In [ ]:
# ============================================================
# 9. Prepare comparison blocks
# ============================================================

reference_comparison_df["_reference_index"] = np.arange(
    len(reference_comparison_df)
)

extracted_comparison_df["_extraction_index"] = np.arange(
    len(extracted_comparison_df)
)

for frame in [
    reference_comparison_df,
    extracted_comparison_df,
]:
    frame["_block_category"] = frame["Category"].map(
        identity_text
    )

    frame["_block_source_location"] = frame[
        "Source Location"
    ].map(identity_text)

reference_block_counts = (
    reference_comparison_df[
        ["_block_category", "_block_source_location"]
    ]
    .value_counts()
    .to_dict()
)

extracted_block_counts = (
    extracted_comparison_df[
        ["_block_category", "_block_source_location"]
    ]
    .value_counts()
    .to_dict()
)

print("Reference matching blocks:", len(reference_block_counts))
print("Extraction matching blocks:", len(extracted_block_counts))


In [ ]:
# ============================================================
# 10. Identity-only matching score
# ============================================================

def matching_score(reference_row, extracted_row):

    topic_score = text_similarity(
        reference_row["Topic"],
        extracted_row["Topic"],
    )

    description_score = text_similarity(
        reference_row["Description"],
        extracted_row["Description"],
    )

    period_score = text_similarity(
        reference_row["Reporting Period"],
        extracted_row["Reporting Period"],
    )

    total_score = (
        MATCHING_WEIGHTS["topic"] * topic_score
        + MATCHING_WEIGHTS["description"] * description_score
        + MATCHING_WEIGHTS["reporting_period"] * period_score
    )

    return {
        "total": total_score,
        "topic": topic_score,
        "description": description_score,
        "period": period_score,
    }


print(
    "Alignment uses Category + Source Location blocking, then "
    "Topic + Description + Reporting Period identity evidence."
)
print("Value, Unit and Qualifier are excluded from alignment.")


In [ ]:
# ============================================================
# 11. One-to-one Hungarian record alignment
# ============================================================

matched_pairs = []
matched_reference_indices = set()
matched_extraction_indices = set()

reference_blocks = reference_comparison_df.groupby(
    ["_block_category", "_block_source_location"],
    dropna=False,
)

for block_key, reference_block in reference_blocks:

    category_key, source_key = block_key

    extracted_block = extracted_comparison_df.loc[
        (
            extracted_comparison_df["_block_category"]
            == category_key
        )
        & (
            extracted_comparison_df["_block_source_location"]
            == source_key
        )
    ]

    if extracted_block.empty:
        continue

    reference_rows = list(reference_block.iterrows())
    extracted_rows = list(extracted_block.iterrows())

    score_matrix = np.zeros(
        (len(reference_rows), len(extracted_rows)),
        dtype=float,
    )

    component_scores = {}

    for i, (_, reference_row) in enumerate(reference_rows):
        for j, (_, extracted_row) in enumerate(extracted_rows):
            scores = matching_score(
                reference_row,
                extracted_row,
            )

            score_matrix[i, j] = scores["total"]
            component_scores[(i, j)] = scores

    row_indices, column_indices = linear_sum_assignment(
        -score_matrix
    )

    for row_i, column_j in zip(
        row_indices,
        column_indices,
    ):
        score = float(score_matrix[row_i, column_j])

        if score < MATCH_SCORE_THRESHOLD:
            continue

        reference_row = reference_rows[row_i][1]
        extracted_row = extracted_rows[column_j][1]
        scores = component_scores[(row_i, column_j)]

        reference_index = int(
            reference_row["_reference_index"]
        )

        extraction_index = int(
            extracted_row["_extraction_index"]
        )

        matched_pairs.append({
            "reference_index": reference_index,
            "extraction_index": extraction_index,
            "matching_score": score,
            "topic_matching_score": scores["topic"],
            "description_matching_score": scores["description"],
            "period_matching_score": scores["period"],
        })

        matched_reference_indices.add(reference_index)
        matched_extraction_indices.add(extraction_index)


matched_pairs = sorted(
    matched_pairs,
    key=lambda item: item["reference_index"],
)

print("Aligned records:", len(matched_pairs))


In [ ]:
# ============================================================
# 12. Missing and unsupported/unmatched record tables
# ============================================================

all_reference_indices = set(
    reference_comparison_df["_reference_index"].astype(int)
)

all_extraction_indices = set(
    extracted_comparison_df["_extraction_index"].astype(int)
)

missing_reference_indices = sorted(
    all_reference_indices - matched_reference_indices
)

unsupported_extraction_indices = sorted(
    all_extraction_indices - matched_extraction_indices
)

missing_records_df = (
    reference_comparison_df.loc[
        reference_comparison_df["_reference_index"].isin(
            missing_reference_indices
        ),
        FIELDS + ["_reference_index"],
    ]
    .rename(columns={"_reference_index": "Reference Index"})
    .reset_index(drop=True)
)

unsupported_records_df = (
    extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"].isin(
            unsupported_extraction_indices
        ),
        FIELDS + ["_extraction_index"],
    ]
    .rename(columns={"_extraction_index": "Extraction Index"})
    .reset_index(drop=True)
)

print("Missing reference records:", len(missing_records_df))
print(
    "Unsupported/unmatched extracted records:",
    len(unsupported_records_df),
)


In [ ]:
# ============================================================
# 13. Field-level comparison of aligned records
# ============================================================

comparison_rows = []

for pair in matched_pairs:

    reference_row = reference_comparison_df.loc[
        reference_comparison_df["_reference_index"]
        == pair["reference_index"]
    ].iloc[0]

    extracted_row = extracted_comparison_df.loc[
        extracted_comparison_df["_extraction_index"]
        == pair["extraction_index"]
    ].iloc[0]

    field_matches = {
        "Category": exact_normalised_text_match(
            reference_row["Category"],
            extracted_row["Category"],
        ),

        "Topic": topic_correct(
            reference_row["Topic"],
            extracted_row["Topic"],
        ),

        "Description": exact_normalised_text_match(
            reference_row["Description"],
            extracted_row["Description"],
        ),

        "Value": value_correct(
            reference_row["Value"],
            extracted_row["Value"],
        ),

        "Unit": unit_correct(
            reference_row["Unit"],
            extracted_row["Unit"],
        ),

        "Qualifier": qualifier_correct(
            reference_row["Qualifier"],
            extracted_row["Qualifier"],
        ),

        "Reporting Period": period_correct(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"],
        ),

        "Source Location": exact_normalised_text_match(
            reference_row["Source Location"],
            extracted_row["Source Location"],
        ),
    }

    all_mismatched_fields = [
        field
        for field in FIELDS
        if not field_matches[field]
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    fully_correct = (
        len(primary_mismatched_fields) == 0
    )

    output_row = {
        "Reference Index": pair["reference_index"],
        "Extraction Index": pair["extraction_index"],
        "Category": reference_row["Category"],
        "Matching Score": pair["matching_score"],
        "Topic Matching Score": pair["topic_matching_score"],
        "Description Matching Score":
            pair["description_matching_score"],
        "Reporting Period Matching Score":
            pair["period_matching_score"],
        "Description Lexical Similarity":
            text_similarity(
                reference_row["Description"],
                extracted_row["Description"],
            ),
        "Fully Correct": bool(fully_correct),
        "all_mismatched_fields":
            ", ".join(all_mismatched_fields),
        "primary_mismatched_fields":
            ", ".join(primary_mismatched_fields),
    }

    for field in FIELDS:
        output_row[f"Reference {field}"] = reference_row[field]
        output_row[f"Extracted {field}"] = extracted_row[field]
        output_row[f"{field} Match"] = bool(
            field_matches[field]
        )

    comparison_rows.append(output_row)


comparison_df = pd.DataFrame(comparison_rows)

print("Compared aligned records:", len(comparison_df))

if not comparison_df.empty:
    print(
        "Fully correct primary records:",
        int(comparison_df["Fully Correct"].sum()),
    )

display(comparison_df.head(10))


In [ ]:
# ============================================================
# 14. Split fully correct and discrepant aligned records
# ============================================================

if comparison_df.empty:
    fully_correct_records_df = comparison_df.copy()
    discrepant_records_df = comparison_df.copy()
else:
    fully_correct_records_df = comparison_df.loc[
        comparison_df["Fully Correct"]
    ].copy()

    discrepant_records_df = comparison_df.loc[
        ~comparison_df["Fully Correct"]
    ].copy()

print("Fully correct aligned records:", len(fully_correct_records_df))
print("Discrepant aligned records:", len(discrepant_records_df))

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Category",
                "primary_mismatched_fields",
            ]
        ].reset_index(drop=True)
    )


In [ ]:
# ============================================================
# 15. Field-level validation and error summary
# ============================================================

field_rows = []

for field in FIELDS:

    if comparison_df.empty:
        correct_count = 0
        evaluated_count = 0
        accuracy = None
    else:
        evaluated_count = len(comparison_df)
        correct_count = int(
            comparison_df[f"{field} Match"].sum()
        )

        accuracy = (
            correct_count / evaluated_count
            if evaluated_count > 0
            else None
        )

    role = (
        "diagnostic"
        if field == DESCRIPTION_DIAGNOSTIC_FIELD
        else "primary"
    )

    field_rows.append({
        "Field": field,
        "Role": role,
        "Aligned Records": evaluated_count,
        "Correct Records": correct_count,
        "Incorrect Records":
            evaluated_count - correct_count,
        "Accuracy": accuracy,
    })


field_validation_df = pd.DataFrame(field_rows)

field_error_summary_df = field_validation_df.loc[
    field_validation_df["Incorrect Records"] > 0
].copy()

display(field_validation_df)


In [ ]:
# ============================================================
# 16. Calculate common validation metrics
# ============================================================

reference_record_count = int(
    len(reference_comparison_df)
)

extracted_record_count = int(
    len(extracted_comparison_df)
)

fully_correct_record_count = int(
    comparison_df["Fully Correct"].sum()
)

discrepant_record_count = (
    aligned_record_count
    - fully_correct_record_count
)

completeness = (
    aligned_record_count / reference_record_count
    if reference_record_count
    else 0.0
)

missing_rate = (
    missing_record_count / reference_record_count
    if reference_record_count
    else 0.0
)

record_precision_exact = (
    fully_correct_record_count / extracted_record_count
    if extracted_record_count
    else 0.0
)

record_recall_exact = (
    fully_correct_record_count / reference_record_count
    if reference_record_count
    else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if (record_precision_exact + record_recall_exact)
    else 0.0
)

unsupported_rate = (
    unsupported_record_count / extracted_record_count
    if extracted_record_count
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count / aligned_record_count
    if aligned_record_count
    else 0.0
)

correct_primary_field_instances = int(
    sum(
        comparison_df[f"{field} Match"].sum()
        for field in PRIMARY_CORRECTNESS_FIELDS
    )
)

expected_primary_field_instances = int(
    reference_record_count
    * len(PRIMARY_CORRECTNESS_FIELDS)
)

field_accuracy = (
    correct_primary_field_instances
    / expected_primary_field_instances
    if expected_primary_field_instances
    else 0.0
)

description_diagnostic_accuracy = (
    float(
        comparison_df[
            f"{DESCRIPTION_DIAGNOSTIC_FIELD} Match"
        ].mean()
    )
    if aligned_record_count
    else None
)

print("Fully correct records:", fully_correct_record_count)
print("Discrepant records:", discrepant_record_count)
print("Completeness:", completeness)
print("Exact record precision:", record_precision_exact)
print("Exact record recall:", record_recall_exact)
print("Exact record F1:", record_f1_exact)
print("Field accuracy:", field_accuracy)
print(
    "Description diagnostic accuracy:",
    description_diagnostic_accuracy
)


In [ ]:
# ============================================================
# 17. Category-level metrics
# ============================================================

category_rows = []

for category in EXPECTED_CATEGORY_COUNTS:

    expected_records = int(
        (reference_df["Category"] == category).sum()
    )

    extracted_records_category = sum(
        1
        for record in extracted_records
        if record.get("Category") == category
    )

    category_comparison = (
        comparison_df.loc[
            comparison_df["Category"] == category
        ]
        if not comparison_df.empty
        else comparison_df
    )

    aligned_records_category = len(category_comparison)

    fully_correct_category = (
        int(category_comparison["Fully Correct"].sum())
        if not category_comparison.empty
        else 0
    )

    discrepant_category = (
        aligned_records_category - fully_correct_category
    )

    category_completeness = (
        aligned_records_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_precision = (
        fully_correct_category / extracted_records_category
        if extracted_records_category > 0
        else 0.0
    )

    category_recall = (
        fully_correct_category / expected_records
        if expected_records > 0
        else 0.0
    )

    category_f1 = (
        2 * category_precision * category_recall
        / (category_precision + category_recall)
        if category_precision + category_recall > 0
        else 0.0
    )

    category_rows.append({
        "Category": category,
        "Expected Records": expected_records,
        "Extracted Records": extracted_records_category,
        "Aligned Records": aligned_records_category,
        "Fully Correct Records": fully_correct_category,
        "Discrepant Records": discrepant_category,
        "Completeness": category_completeness,
        "Record Precision Exact": category_precision,
        "Record Recall Exact": category_recall,
        "Record F1 Exact": category_f1,
    })


category_metrics_df = pd.DataFrame(category_rows)
display(category_metrics_df)


In [ ]:
# ============================================================
# 18. Field-level accuracy diagnostics
# ============================================================

field_accuracy_rows = []

for field in FIELDS:
    correct_count = int(
        comparison_df[f"{field} Match"].sum()
    )

    aligned_count = len(comparison_df)

    field_accuracy_rows.append({
        "Field": field,
        "Used in Alignment Block":
            field in BLOCK_FIELDS,
        "Used in Primary Correctness":
            field in PRIMARY_CORRECTNESS_FIELDS,
        "Diagnostic Only":
            field == DESCRIPTION_DIAGNOSTIC_FIELD,
        "Correct Records": correct_count,
        "Aligned Records": aligned_count,
        "Accuracy Among Aligned": (
            correct_count / aligned_count
            if aligned_count
            else None
        ),
        "Overall Accuracy Against Reference": (
            correct_count / reference_record_count
            if reference_record_count
            else None
        ),
    })

field_accuracy_df = pd.DataFrame(
    field_accuracy_rows
)

field_accuracy_dictionary = {
    row["Field"]: (
        float(row["Accuracy Among Aligned"])
        if pd.notna(row["Accuracy Among Aligned"])
        else None
    )
    for _, row in field_accuracy_df.iterrows()
}

display(field_accuracy_df)


In [ ]:
# ============================================================
# 19. Build final Branch B validation summary
# ============================================================

category_metrics_dictionary = {
    row["Category"]: {
        "expected_records":
            int(row["Expected Records"]),
        "extracted_records":
            int(row["Extracted Records"]),
        "aligned_records":
            int(row["Aligned Records"]),
        "fully_correct_records":
            int(row["Fully Correct Records"]),
        "discrepant_records":
            int(row["Discrepant Records"]),
        "completeness":
            float(row["Completeness"]),
        "record_precision_exact":
            float(row["Record Precision Exact"]),
        "record_recall_exact":
            float(row["Record Recall Exact"]),
        "record_f1_exact":
            float(row["Record F1 Exact"]),
    }
    for _, row in category_metrics_df.iterrows()
}

summary = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "reference_records": reference_record_count,
    "extracted_records": extracted_record_count,
    "aligned_records": int(aligned_record_count),
    "fully_correct_records":
        fully_correct_record_count,
    "discrepant_records":
        discrepant_record_count,
    "missing_records":
        int(missing_record_count),
    "unsupported_extracted_records":
        int(unsupported_record_count),

    "completeness":
        round(completeness, 4),
    "missing_rate":
        round(missing_rate, 4),
    "record_precision_exact":
        round(record_precision_exact, 4),
    "record_recall_exact":
        round(record_recall_exact, 4),
    "record_f1_exact":
        round(record_f1_exact, 4),
    "unsupported_rate":
        round(unsupported_rate, 4),
    "discrepancy_rate_among_aligned":
        round(discrepancy_rate_among_aligned, 4),
    "field_accuracy":
        round(field_accuracy, 4),

    "description_diagnostic_accuracy": (
        None
        if description_diagnostic_accuracy is None
        else round(
            description_diagnostic_accuracy,
            4
        )
    ),

    "field_accuracy_among_aligned":
        field_accuracy_dictionary,

    "schema_validity":
        schema_validity,
    "schema_diagnostics":
        schema_diagnostics,
    "structurally_evaluable":
        structurally_evaluable,
    "content_diagnostics":
        content_diagnostics,

    "alignment_block_fields":
        BLOCK_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "description_status":
        (
            "Diagnostic only; excluded from primary "
            "record correctness exactly as frozen in "
            "final D8 Validation A."
        ),

    "matching_rules": {
        "blocking_fields":
            BLOCK_FIELDS,
        "one_to_one_assignment":
            "Hungarian linear-sum assignment",
        "matching_score_threshold":
            MATCH_SCORE_THRESHOLD,
        "matching_score_weights":
            MATCHING_WEIGHTS,
        "value_used_for_alignment":
            False,
        "unit_used_for_alignment":
            False,
        "qualifier_used_for_alignment":
            False,
        "description_used_for_alignment":
            True,
        "description_used_for_primary_correctness":
            False,
    },

    "comparison_rules_frozen_from_branch_A":
        True,

    "normalisation_note":
        (
            "Deterministic normalisation is applied only "
            "to comparison copies; the preserved Branch B "
            "extraction is not modified."
        ),

    "category_metrics":
        category_metrics_dictionary,

    "input_provenance":
        input_provenance,
}

print(json.dumps(
    summary,
    indent=2,
    ensure_ascii=False
))


In [ ]:
# ============================================================
# 20. Validation integrity checks
# ============================================================

assert (
    aligned_record_count
    + missing_record_count
    == reference_record_count
)

assert (
    aligned_record_count
    + unsupported_record_count
    == extracted_record_count
)

assert (
    fully_correct_record_count
    + discrepant_record_count
    == aligned_record_count
)

for metric_name, metric_value in {
    "completeness": completeness,
    "missing_rate": missing_rate,
    "record_precision_exact":
        record_precision_exact,
    "record_recall_exact":
        record_recall_exact,
    "record_f1_exact":
        record_f1_exact,
    "unsupported_rate":
        unsupported_rate,
    "discrepancy_rate":
        discrepancy_rate_among_aligned,
    "field_accuracy":
        field_accuracy,
}.items():
    assert 0.0 <= metric_value <= 1.0, (
        f"Invalid {metric_name}: {metric_value}"
    )

print("Validation integrity checks passed.")


In [ ]:
# ============================================================
# 21. Export validation artefacts
# ============================================================

DETAILED_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_validation_detailed.csv"
)

FULLY_CORRECT_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_fully_correct_records.csv"
)

DISCREPANT_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_discrepant_records.csv"
)

MISSING_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_missing_records.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_unsupported_records.csv"
)

FIELD_VALIDATION_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_field_validation.csv"
)

FIELD_ERROR_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_field_error_summary.csv"
)

FIELD_ACCURACY_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_field_accuracy.csv"
)

CATEGORY_METRICS_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_category_metrics.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "D8_branch_B_validation_summary.json"
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_PATH,
    index=False,
    encoding="utf-8-sig"
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig"
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig"
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_validation_df.to_csv(
    FIELD_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_error_summary_df.to_csv(
    FIELD_ERROR_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_accuracy_df.to_csv(
    FIELD_ACCURACY_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("D8 Validation B artefacts saved.")


In [ ]:
# ============================================================
# 22. Download generated validation artefacts
# ============================================================

GENERATED_OUTPUTS = [
    DETAILED_PATH,
    FULLY_CORRECT_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    FIELD_VALIDATION_PATH,
    FIELD_ERROR_SUMMARY_PATH,
    FIELD_ACCURACY_PATH,
    CATEGORY_METRICS_PATH,
    SUMMARY_PATH,
]

for output_path in GENERATED_OUTPUTS:
    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

for output_path in GENERATED_OUTPUTS:
    if output_path.exists():
        files.download(output_path)
